# Прокси пола, дисбаланс, калибровка и порог (модель «только объяснимые признаки»)
Нужны: models/features_explainable.json из 09, обновлённые src/models.py, src/metrics.py,
src/calibration.py, src/decision.py.
Всё на dev, hold-out не трогаем.

План:
0. Может ли модель без пола восстановить его по другим признакам (прокси)?
1. Три способа обращения с дисбалансом: без изменений, веса классов, undersampling.
2. Калибровка вероятностей (Platt и изотоническая) и её проверка.
3. Порог отказа по стоимости ошибок.

In [ ]:
import json
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score

sys.path.insert(0, "..")
from src.calibration import Calibrator, crossfit_calibrate
from src.data import load_features, split_dev_holdout
from src.decision import best_threshold, loss_curve
from src.features import ID_COL, TARGET_COL
from src.metrics import calibration_report, calibration_table
from src.models import RANDOM_STATE, cv_auc, cv_predict, make_lgbm, to_lgbm

PROCESSED = Path("../data/processed")
MODELS = Path("../models")
pd.set_option("display.width", 140)

df = load_features(PROCESSED / "train_features.parquet")
dev, _ = split_dev_holdout(df, PROCESSED / "holdout_ids.csv")  # hold-out отбрасываем
y = dev[TARGET_COL]
cfg = json.loads((MODELS / "lgbm_best_params.json").read_text())
features = json.loads((MODELS / "features_explainable.json").read_text())
X = to_lgbm(dev[features])
print("dev:", X.shape, f"| доля дефолтов {y.mean():.4f}")


def make_model(**extra):
    return make_lgbm(n_estimators=cfg["n_estimators"], learning_rate=cfg["learning_rate"],
                     **cfg["params"], **extra)

## 0. Прокси пола
После удаления `CODE_GENDER` пол всё равно может проникать через другие признаки.
Проверка 1: связь доли женщин в категории профессии или организации с долей дефолтов.
Проверка 2: насколько хорошо пол восстанавливается по оставшимся признакам (AUC 0.5 = никак).

In [ ]:
known = dev[dev["CODE_GENDER"].notna()]
for col in ["ORGANIZATION_TYPE", "OCCUPATION_TYPE"]:
    t = known.groupby(col).agg(
        n=(TARGET_COL, "size"),
        share_female=("CODE_GENDER", lambda s: (s == "F").mean()),
        default=(TARGET_COL, "mean"),
    )
    t = t[t["n"] >= 200].sort_values("share_female")
    print(f"\n--- {col}: меньше всего женщин ---")
    print(t.head(6).round(3))
    print(f"--- {col}: больше всего женщин ---")
    print(t.tail(6).round(3))
    print("Спирмен (доля женщин, доля дефолтов) по категориям:",
          round(t["share_female"].corr(t["default"], method="spearman"), 3))

In [ ]:
sub = dev[dev["CODE_GENDER"].notna()].sample(100_000, random_state=RANDOM_STATE)
y_g = (sub["CODE_GENDER"] == "F").astype(int).reset_index(drop=True)
X_g = to_lgbm(sub[features]).reset_index(drop=True)
auc_g = cv_auc(make_lgbm(n_estimators=200, learning_rate=0.1), X_g, y_g, n_splits=3)
print(f"AUC восстановления пола по объяснимым признакам: {auc_g.mean():.3f} (по фолдам {auc_g.round(3)})")

## 1. Дисбаланс: три варианта
Все варианты обучаются с теми же гиперпараметрами и числом деревьев; для undersampling
и весов число деревьев заново не подбиралось (это ограничение сравнения).
SMOTE не берём: в данных пропуски и категории, для него пришлось бы кодировать и заполнять признаки.

In [ ]:
pos_weight = (y == 0).sum() / (y == 1).sum()
print(f"Вес положительного класса при полной балансировке: {pos_weight:.1f}")


def undersample(ratio, seed=RANDOM_STATE):
    """Оставляем всех дефолтных и ratio * их число недефолтных, только в обучающей части фолда."""
    def f(tr, y_):
        rng = np.random.default_rng(seed)
        labels = y_.iloc[tr].values
        pos, neg = tr[labels == 1], tr[labels == 0]
        keep = rng.choice(neg, size=min(len(neg), ratio * len(pos)), replace=False)
        return np.sort(np.concatenate([pos, keep]))
    return f


variants = {
    "baseline": dict(make=lambda: make_model(), filt=None),
    "scale_pos_weight": dict(make=lambda: make_model(scale_pos_weight=pos_weight), filt=None),
    "undersample 1:3": dict(make=lambda: make_model(), filt=undersample(3)),
}
oofs = {}
for name, v in variants.items():
    oof, fa = cv_predict(v["make"], X, y, train_filter=v["filt"])
    oofs[name] = oof
    print(f"{name}: AUC по фолдам {fa.mean():.4f} ± {fa.std():.4f}")

In [ ]:
# Ранжирование и калибровка: до и после Platt (кросс-подгонка, без утечки)
rows = []
for name, oof in oofs.items():
    for cal, p in [("raw", oof), ("platt", crossfit_calibrate(oof, y, "platt"))]:
        r = calibration_report(name, y, p)
        r["calibration"] = cal
        r["ROC AUC"] = round(roc_auc_score(y, p), 4)
        r["PR AUC"] = round(average_precision_score(y, p), 4)
        rows.append(r)
pd.DataFrame(rows).set_index(["model", "calibration"])

## 2. Калибровка базовой модели

In [ ]:
p_base = oofs["baseline"]
calibrated = {"raw": p_base,
              "platt": crossfit_calibrate(p_base, y, "platt"),
              "isotonic": crossfit_calibrate(p_base, y, "isotonic")}
pd.DataFrame([calibration_report(k, y, v) for k, v in calibrated.items()]).set_index("model")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for name, p in calibrated.items():
    t = calibration_table(y, p, bins=10)
    axes[0].plot(t["predicted"], t["observed"], "o-", label=name)
    t20 = calibration_table(y, p, bins=20)
    axes[1].plot(t20["predicted"], t20["observed"], "o-", label=name)
for ax, lim in zip(axes, [1.0, 0.25]):
    ax.plot([0, lim], [0, lim], "k--", alpha=0.4)
    ax.set(xlim=(0, lim), ylim=(0, lim), xlabel="средний прогноз", ylabel="фактическая доля дефолтов")
    ax.legend()
axes[0].set_title("Калибровка (10 квантильных корзин)")
axes[1].set_title("Калибровка, увеличенный масштаб (20 корзин)")
plt.tight_layout()
plt.show()

## 3. Порог по стоимости ошибок
Проводим расчёт для откалиброванных вероятностей лучшего метода (по Brier).
Единица стоимости: отказ добросовестному клиенту = 1, кредит будущему неплательщику = R.
Для идеально откалиброванных вероятностей оптимальный порог равен 1 / (1 + R).
R здесь предположение, а не данные: у реальной компании оно считается из процентной маржи и доли потерь.

In [ ]:
best_method = min(["platt", "isotonic"], key=lambda m: brier_score_loss(y, calibrated[m]))
p_cal = calibrated[best_method]
print("Лучший метод калибровки по Brier:", best_method)

pos_rate = y.mean()
rows = []
for R in [3, 5, 8, 10]:
    b = best_threshold(loss_curve(y, p_cal, cost_fn=R, cost_fp=1))
    naive = min(R * pos_rate, 1 - pos_rate)  # лучшая из стратегий «одобрять всех» / «отказывать всем»
    rows.append({
        "R = C_FN / C_FP": R,
        "теор. порог 1/(1+R)": round(1 / (1 + R), 3),
        "лучший порог": round(b.threshold, 3),
        "доля одобрений": round(b.approval_rate, 3),
        "отсеяно дефолтов": round(b.recall, 3),
        "потери на клиента": round(b.loss, 3),
        "потери наивной стратегии": round(naive, 3),
        "экономия, %": round((1 - b.loss / naive) * 100, 1),
    })
pd.DataFrame(rows).set_index("R = C_FN / C_FP")

In [ ]:
R = 5
curve = loss_curve(y, p_cal, cost_fn=R, cost_fp=1)
c = curve[np.isfinite(curve["threshold"])]
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(c["threshold"], c["loss"])
ax.axvline(1 / (1 + R), color="k", ls="--", alpha=0.5, label=f"теоретический порог 1/(1+R) = {1 / (1 + R):.3f}")
ax.set(xlim=(0, 0.5), xlabel="порог отказа (откалиброванная вероятность)",
       ylabel="потери на клиента", title=f"Потери в зависимости от порога, R = {R}")
ax.legend()
plt.show()

## 4. Сохраняем

In [ ]:
oof_df = pd.DataFrame({ID_COL: dev[ID_COL], TARGET_COL: y})
for name, oof in oofs.items():
    oof_df[name] = oof
oof_df[f"baseline_{best_method}"] = p_cal
oof_df.to_parquet(PROCESSED / "oof_explainable.parquet", index=False)

# калибратор для бота: обучен на всех OOF-прогнозах baseline
joblib.dump(Calibrator(best_method).fit(p_base, y), MODELS / "calibrator.joblib")
print("Сохранено: oof_explainable.parquet, calibrator.joblib")

Что смотрим:
1. восстанавливается ли пол по оставшимся признакам (если AUC заметно выше 0.5, прокси есть);
2. меняют ли веса и undersampling AUC и PR AUC, и что они делают со средним прогнозом;
3. насколько ECE и Brier улучшаются после калибровки и какой метод лучше;
4. совпадает ли лучший порог с теоретическим 1/(1+R) и как он зависит от R.